# Phase 3 RR - Architecture 3 (linear-attention hybrid) + QLoRA (4-bit NF4 frozen backbone + LoRA)

**Task:** rhetorical-role prediction. For every sentence of a legal document (Corporate Law *CL* and Information Technology *IT*) predict exactly one of the 13 roles.  
**Phase-1 model:** Architecture 3 (linear-attention hybrid) - mostly recurrent linear-attention layers; the 4-bit weights also shrink the memory footprint.  
**Default adaptation:** **QLoRA (4-bit NF4 frozen backbone + LoRA)**; the other methods are one `METHOD = ...` switch away.

**This notebook is self-contained** - run the cells top to bottom. You normally only edit the *Configuration* cell.

| step | what happens |
|---|---|
| 1-3 | setup, configuration, your Phase-1 tokenizer |
| 4-6 | load CL / IT data, sentence tokens, context windows, class balance |
| 7 | metrics (macro-F1/P/R) and error analysis |
| 8-10 | load your Phase-1 model, adaptation methods, sentence classifier |
| 11-12 | training code, then run: train, validate, predict test, write `submission.zip` |

*Cells labelled `# helper` are plumbing (parsing, metrics, your unchanged Phase-1 code); you never need to open them.*

**How a prediction is made.** The sentences of a document are packed into windows (a few hundred tokens of *core* sentences plus up to 256 tokens of preceding *context* sentences). The Phase-1 model reads the window, every sentence is turned into one vector (mean of its token states + state at its last token + a domain embedding), a BiGRU over those vectors lets neighbouring sentences influence each other (document context), and a linear layer gives the role. `CONTEXT = 'sentence'` removes both the context sentences and the BiGRU (sentence-only baseline).

**Before running:** attach your Phase-1 checkpoint dataset and `kratikasum/tokenizer` as Kaggle inputs and keep the secret `HF`. Run once with `SMOKE_TEST = True`, then switch it off. Only Phase-1 weights are used.

**Outputs** (in `/kaggle/working/<RUN_NAME>/`): `prediction.json` (+ `CL_prediction.json`, `IT_prediction.json`), `submission.zip`, `results.json`, saved trainable weights.

## 1. Setup

In [10]:
import os, re, sys, json, glob, math, time, random, zipfile, gc, pickle, copy
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict, fields
from typing import List, Dict, Tuple, Optional
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import login, hf_hub_download
from tqdm.auto import tqdm

try:                                                     # Kaggle secret 'HF' (same as in your Phase-1/2 notebooks)
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
except Exception as ex:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    print(f"Kaggle secret 'HF' not available ({type(ex).__name__}); using env HF_TOKEN / local files")

print('CUDA available:', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}:', torch.cuda.get_device_name(i))
N_GPU = torch.cuda.device_count()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
use_bf16 = device == 'cuda' and torch.cuda.get_device_capability()[0] >= 8      # T4/P100: fp16 + GradScaler
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


CUDA available: True
GPU 0: Tesla T4
GPU 1: Tesla T4


## 2. Configuration
Every setting of the experiment is here.

In [11]:
# ---------- where things are ----------
HF_DATASET_REPO = 'Exploration-Lab/CS-781-Capstone'
PHASE3_DIR      = 'Phase_3_RR'          # folder with the task data inside that dataset
LOCAL_DATA_DIR  = None                  # or e.g. '/kaggle/input/<dataset>/Phase_3_RR'
PHASE1_CKPT     = ''                    # your Phase-1 best.pt / last.pt   ('' = search /kaggle/input)
CKPT_HINT       = 'arch3'                # used by that search: path must contain this text ('-' and '_' are ignored)
RUN_NAME        = 'p3_arch3_lorma'
OUT_ROOT        = '/kaggle/working' if os.path.isdir('/kaggle/working') else './outputs'

# ---------- data handling ----------
SEED            = 42
DOMAINS         = ('CL', 'IT')          # one model is trained on both domains (a learned domain embedding tells them apart)
CONTEXT         = 'document'            # 'document' = sentence + surrounding sentences + BiGRU over sentence vectors | 'sentence' = each sentence alone
RUN_CONTEXT_ABLATION = False            # True = ALSO train the other context mode and compare (doubles the run time)
CORE_TOK, CTX_TOK = 768, 256              # tokens whose sentences are predicted per window / left-context tokens given before them
MAX_SENT_TOK    = 96                    # a sentence is cut after this many tokens
CW_POWER        = 0.5                   # class weight = (N / (C * count_label)) ** CW_POWER   (0 = off)
LABEL_SMOOTH    = 0.05

# ---------- adaptation method ----------
METHOD = 'lorma'                         # 'qlora' | 'lora' | 'lorma' | 'full'   (prefix tuning is not defined for the linear-attention layers)
LORA_R, LORA_ALPHA, LORA_DROPOUT = 32, 64, 0.05            # 'lora' / 'lorma' / 'qlora': rank, scale, dropout
PEFT_TARGETS = ('q', 'k', 'v', 'g', 'o', 'gate', 'up', 'down')   # linears that get LoRA / LoRMA / QLoRA matrices
QLORA_BLOCK = 64                        # 'qlora': NF4 quantisation block size
N_PREFIX = 32                           # 'prefix': learned key/value vectors per attention layer
HEAD_HIDDEN, GRU_LAYERS = 256, 2        # sentence-level head

# ---------- training ----------
EPOCHS, LR, HEAD_LR = 9, 3e-4, 1e-3        # LR = adapted backbone parameters (prefix tuning needs ~5e-3, LoRA-style methods ~3e-4)
BATCH_SIZE, GRAD_ACCUM = 4, 4           # windows per step x accumulation  ('sentence' mode uses 8x larger batches)
PATIENCE = 3                            # stop after this many epochs without validation improvement
GRAD_CKPT = False                       # True = less GPU memory, ~30% slower

# ---------- quick dry run ----------
SMOKE_TEST = False                      # True = a handful of documents, 6 steps: checks the whole pipeline in minutes
MAX_DOCS = MAX_STEPS = None
if SMOKE_TEST:
    MAX_DOCS, MAX_STEPS = 4, 6
    RUN_NAME += '_smoke'
    EPOCHS = 1

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
OUTPUT_DIR = os.path.join(OUT_ROOT, RUN_NAME)
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 3. Your Phase-1 tokenizer
The class is your Phase-1 code, unchanged.

In [12]:
from typing import List, Dict, Tuple
class ByteLevelBPE:
    """Your tokenizer class (training code removed: we load the already-trained pickle)."""
    def __init__(self):
        self.vocab: Dict[bytes, int] = {}
        self.inverse_vocab: Dict[int, bytes] = {}
        self.merges: Dict[Tuple[bytes, bytes], Tuple[bytes, int]] = {}
        self.word_end_marker = b'</w>'
        self.reserved_tokens = [b'<pad>', b'<unk>', b'<bos>', b'<eos>']

    def tokenize(self, text: str) -> List[bytes]:
        words = text.split()
        tokens = []
        for w in words:
            word_bytes = w.encode('utf-8')
            split = [bytes([b]) for b in word_bytes] + [self.word_end_marker]
            while True:
                best_pair, best_rank = None, float('inf')
                for i in range(len(split) - 1):
                    pair = (split[i], split[i+1])
                    if pair in self.merges:
                        rank = self.merges[pair][1]
                        if rank < best_rank:
                            best_rank, best_pair = rank, pair
                if best_pair is None:
                    break
                new_split, i = [], 0
                while i < len(split):
                    if i < len(split) - 1 and split[i] == best_pair[0] and split[i+1] == best_pair[1]:
                        new_split.append(self.merges[best_pair][0])
                        i += 2
                    else:
                        new_split.append(split[i])
                        i += 1
                split = new_split
            tokens.extend(split)
        return tokens

    def encode(self, text: str) -> List[int]:
        unk_id = self.vocab[b'<unk>']
        return [self.vocab.get(token, unk_id) for token in self.tokenize(text)]

    def decode(self, token_ids: List[int]) -> str:
        skip_ids = {self.vocab[t] for t in self.reserved_tokens if t in self.vocab}
        bytes_list = []
        for tid in token_ids:
            if tid in skip_ids:
                continue
            bytes_list.append(self.inverse_vocab.get(tid, b''))
        joined = b''.join(bytes_list)
        return joined.replace(self.word_end_marker, b' ').decode('utf-8', errors='replace').strip()

    @classmethod
    def load(cls, filepath: str):
        with open(filepath, 'rb') as f:
            data = pickle.load(f)
        bpe = cls()
        bpe.vocab = data['vocab']
        bpe.inverse_vocab = {v: k for k, v in bpe.vocab.items()}
        bpe.merges = data['merges']
        return bpe

    def __len__(self):
        return len(self.vocab)

    @property
    def eos_token_id(self):
        return self.vocab.get(b'<eos>')

In [13]:
cands = glob.glob('/kaggle/input/**/bpe_tokenized_txt.pkl', recursive=True)
if cands:
    tok_path = cands[0]
else:
    import kagglehub
    tok_path = os.path.join(kagglehub.dataset_download('kratikasum/tokenizer'), 'bpe_tokenized_txt.pkl')
tokenizer = ByteLevelBPE.load(tok_path)
VOCAB_SIZE = max(tokenizer.vocab.values()) + 1
print(f'tokenizer: {len(tokenizer):,} entries | embedding size {VOCAB_SIZE:,} | marker {tokenizer.word_end_marker}')

tokenizer: 48,000 entries | embedding size 48,000 | marker b'</w>'


In [14]:
word_cache = {}                          # the Phase-1 tokenizer works word by word, so caching per word is exact
EOS = tokenizer.eos_token_id
PAD_ID = tokenizer.vocab[b'<pad>']


def fast_encode(text):
    out = []
    for w in text.split():
        ids = word_cache.get(w)
        if ids is None:
            ids = word_cache[w] = tokenizer.encode(w)
        out.extend(ids)
    return out

## 4. Load the data
CL and IT training / development documents (ordered sentences + one role per sentence) and the unlabeled test documents.
The role list comes from `label_vocab.json`; every sentence is tokenised with your Phase-1 tokenizer and closed with the `<eos>` token, which acts as a sentence separator.

In [15]:
# helper: file lookup + tolerant JSON parsing
def find_data_file(name, sub):
    """Local override -> attached Kaggle dataset -> Hugging Face dataset repo."""
    if LOCAL_DATA_DIR:
        for p in (os.path.join(LOCAL_DATA_DIR, sub, name), os.path.join(LOCAL_DATA_DIR, name)):
            if os.path.exists(p):
                return p
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if hits:
        return hits[0]
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=f'{PHASE3_DIR}/{sub}/{name}',
                           repo_type='dataset', token=HF_TOKEN)


DEFAULT_LABELS = ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon',
                  'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision',
                  'RulingByPresentCourt', 'None']


def load_label_vocab():
    try:
        raw = json.load(open(find_data_file('label_vocab.json', 'Train'), encoding='utf-8'))
    except Exception as ex:
        print('label_vocab.json not found -> using the 13 labels of the task description:', type(ex).__name__)
        return list(DEFAULT_LABELS)
    if isinstance(raw, dict):
        for k in ('labels', 'label_list', 'id2label', 'label2id'):
            if k in raw:
                raw = raw[k]; break
    if isinstance(raw, dict):
        items = list(raw.items())
        if all(isinstance(v, int) for _, v in items):
            names = [k for k, v in sorted(items, key=lambda kv: kv[1])]
        elif all(str(k).isdigit() for k, _ in items):
            names = [v for k, v in sorted(items, key=lambda kv: int(kv[0]))]
        else:
            names = list(raw.keys())
    else:
        names = [str(x) for x in raw]
    return names or list(DEFAULT_LABELS)


def load_docs(path, domain, labeled, max_docs=None):
    """-> list of dict(uid, id, domain, sents, labels, tok, y). Accepts a list of documents or {doc_id: document}."""
    raw = json.load(open(path, encoding='utf-8'))
    if isinstance(raw, dict):
        items = [(str(k), v) for k, v in raw.items()]
    else:
        items = [(str(d.get('id', i)) if isinstance(d, dict) else str(i), d) for i, d in enumerate(raw)]
    docs, bad = [], Counter()
    for did, d in items[:max_docs]:
        sents = d.get('text', d.get('sentences')) if isinstance(d, dict) else d
        if isinstance(sents, str):
            sents = [sents]
        sents = [str(s) for s in sents]
        y = None
        if labeled:
            labs = d['labels']
            assert len(labs) == len(sents), f'{domain}/{did}: {len(sents)} sentences but {len(labs)} labels'
            y = []
            for l in labs:
                if isinstance(l, (int, np.integer)):
                    y.append(int(l))
                elif str(l).strip() in LAB2ID:
                    y.append(LAB2ID[str(l).strip()])
                else:
                    bad[str(l)] += 1; y.append(LAB2ID.get('None', 0))
            y = np.asarray(y, np.int64)
        tok = [(fast_encode(s)[:MAX_SENT_TOK - 1] + [EOS]) for s in sents]
        docs.append(dict(uid=f'{domain}::{did}', id=did, domain=domain, sents=sents, tok=tok, y=y))
    if bad:
        print('WARNING unknown labels mapped to "None":', dict(bad))
    return docs

### 4a. Roles, documents and statistics

In [16]:
LABELS = load_label_vocab()
LAB2ID = {l: i for i, l in enumerate(LABELS)}
NUM_CLASSES = len(LABELS)
DOM2ID = {d: i for i, d in enumerate(DOMAINS)}
print(NUM_CLASSES, 'rhetorical roles:', LABELS)

train_docs, dev_docs, test_docs = [], [], []
for dom in DOMAINS:
    train_docs += load_docs(find_data_file(f'{dom}_train.json', 'Train'), dom, True, MAX_DOCS)
    dev_docs += load_docs(find_data_file(f'{dom}_dev.json', 'Train'), dom, True, MAX_DOCS)
    test_docs += load_docs(find_data_file(f'{dom}_test.json', 'Test'), dom, False, MAX_DOCS)

for name, ds in (('train', train_docs), ('dev', dev_docs), ('test', test_docs)):
    for dom in DOMAINS:
        sub = [d for d in ds if d['domain'] == dom]
        n_s = sum(len(d['sents']) for d in sub)
        n_t = sum(len(t) for d in sub for t in d['tok'])
        print(f'{name:5s} {dom}: {len(sub):4d} documents | {n_s:7d} sentences | {n_t / max(1, n_s):5.1f} tokens/sentence | '
              f'longest document {max([len(d["sents"]) for d in sub] or [0])} sentences')

13 rhetorical roles: ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']
train CL:   40 documents |   10593 sentences |  35.9 tokens/sentence | longest document 1816 sentences
train IT:   40 documents |    6252 sentences |  36.9 tokens/sentence | longest document 442 sentences
dev   CL:    5 documents |    1396 sentences |  37.2 tokens/sentence | longest document 848 sentences
dev   IT:    5 documents |     746 sentences |  34.3 tokens/sentence | longest document 382 sentences
test  CL:    5 documents |    1339 sentences |  35.3 tokens/sentence | longest document 942 sentences
test  IT:    5 documents |     858 sentences |  38.8 tokens/sentence | longest document 420 sentences


## 5. Context windows and class balance
* A **window** = core sentences (predicted) + preceding context sentences (read only). Core boundaries are re-drawn every epoch.
* Rare roles get a larger loss weight (`CW_POWER`).

In [17]:
# helper: documents -> windows -> batches
def make_windows(di, doc, context, core_tok, ctx_tok):
    """Windows are (doc_index, first_sentence_incl_context, first_core_sentence, end_sentence_exclusive).
    Only the core sentences are predicted / trained; the left-context sentences just give the model more to read."""
    lens = [len(t) for t in doc['tok']]
    n, pos, wins = len(lens), 0, []
    while pos < n:
        if context == 'sentence':
            start, end = pos, pos + 1
        else:
            end, tot = pos, 0
            while end < n and (tot + lens[end] <= core_tok or end == pos):
                tot += lens[end]; end += 1
            start, ctot = pos, 0
            while start > 0 and ctot + lens[start - 1] <= ctx_tok:
                start -= 1; ctot += lens[start]
        wins.append((di, start, pos, end))
        pos = end
    return wins


def bsz(context):
    return BATCH_SIZE if context == 'document' else BATCH_SIZE * 8


def make_batch(docs, wins):
    """-> ids, mask, seg (sentence index of every token, -1 = padding), last (index of each sentence's last token),
       smask (real sentences), dom, lab (-100 = not trained / padding)   all on `device`"""
    B = len(wins)
    S = max(e - s for _, s, _, e in wins)
    T = max(sum(len(t) for t in docs[di]['tok'][s:e]) for di, s, _, e in wins)
    ids = np.full((B, T), PAD_ID, np.int64); mask = np.zeros((B, T), bool); seg = np.full((B, T), -1, np.int64)
    last = np.zeros((B, S), np.int64); smask = np.zeros((B, S), bool)
    lab = np.full((B, S), -100, np.int64); dom = np.zeros(B, np.int64)
    for b, (di, s, c, e) in enumerate(wins):
        d, off = docs[di], 0
        dom[b] = DOM2ID[d['domain']]
        for j, i in enumerate(range(s, e)):
            t = d['tok'][i]; L = len(t)
            ids[b, off:off + L], mask[b, off:off + L], seg[b, off:off + L] = t, True, j
            last[b, j], smask[b, j] = off + L - 1, True
            if i >= c and d['y'] is not None:
                lab[b, j] = d['y'][i]
            off += L
    return tuple(torch.from_numpy(a).to(device) for a in (ids, mask, seg, last, smask, dom, lab))

### 5a. Class weights

In [18]:
counts = np.zeros(NUM_CLASSES)
for d in train_docs:
    counts += np.bincount(d['y'], minlength=NUM_CLASSES)
cw = np.ones(NUM_CLASSES, np.float32)
if CW_POWER > 0:
    cw = ((counts.sum() / (NUM_CLASSES * np.maximum(counts, 1))) ** CW_POWER).astype(np.float32)
    cw = np.minimum(cw / cw[counts > 0].mean(), 8.0)                       # mean weight ~1, rare roles capped at 8
CLASS_W = torch.tensor(cw, device=device)
print('training sentences per role / loss weight:')
for i, l in enumerate(LABELS):
    print(f'   {l:24s} {int(counts[i]):7d}  ({100 * counts[i] / counts.sum():5.1f}%)   weight {cw[i]:.2f}')


training sentences per role / loss weight:
   Fact                        4098  ( 24.3%)   weight 0.19
   Issue                        107  (  0.6%)   weight 1.19
   ArgumentPetitioner          1613  (  9.6%)   weight 0.31
   ArgumentRespondent          1348  (  8.0%)   weight 0.33
   Statute                     1099  (  6.5%)   weight 0.37
   Dissent                      284  (  1.7%)   weight 0.73
   PrecedentReliedUpon         1945  ( 11.5%)   weight 0.28
   PrecedentNotReliedUpon       569  (  3.4%)   weight 0.51
   PrecedentOverruled             3  (  0.0%)   weight 7.08
   RulingByLowerCourt           874  (  5.2%)   weight 0.41
   RatioOfTheDecision          4301  ( 25.5%)   weight 0.19
   RulingByPresentCourt         307  (  1.8%)   weight 0.70
   None                         297  (  1.8%)   weight 0.71


## 6. Metrics and error analysis
* **Macro-F1** (primary), macro precision and macro recall over the roles that occur in the gold labels, per domain and overall;
* the model-selection score is the mean of the CL and IT macro-F1;
* the confusion report shows which closely related roles (e.g. *RatioOfTheDecision* vs *RulingByPresentCourt*, *ArgumentPetitioner* vs *ArgumentRespondent*) are mixed up.

In [19]:
# helper: macro precision / recall / F1, confusion analysis
def scores_from(y_true, y_pred):
    C = NUM_CLASSES
    cm = np.bincount(y_true * C + y_pred, minlength=C * C).reshape(C, C)       # cm[gold, predicted]
    tp, sup, prd = np.diag(cm).astype(float), cm.sum(1), cm.sum(0)
    p, r = tp / np.maximum(prd, 1), tp / np.maximum(sup, 1)
    f = 2 * p * r / np.maximum(p + r, 1e-12)
    act = sup > 0                                                              # macro average over the roles present in the gold labels
    return dict(macro_f1=float(f[act].mean()), macro_p=float(p[act].mean()), macro_r=float(r[act].mean()),
                accuracy=float(tp.sum() / max(1, cm.sum())), n_sentences=int(cm.sum()), cm=cm.tolist(),
                per_label={LABELS[i]: dict(p=float(p[i]), r=float(r[i]), f1=float(f[i]), support=int(sup[i])) for i in range(C)})


def evaluate_docs(docs, probs):
    """-> {'CL': scores, 'IT': scores, 'ALL': scores, 'select': mean of the per-domain macro-F1}"""
    res = {}
    for dom in list(DOMAINS) + ['ALL']:
        ds = [d for d in docs if dom == 'ALL' or d['domain'] == dom]
        if not ds:
            continue
        yt = np.concatenate([d['y'] for d in ds])
        yp = np.concatenate([probs[d['uid']].argmax(1) for d in ds])
        res[dom] = scores_from(yt, yp)
    res['select'] = float(np.mean([res[d]['macro_f1'] for d in DOMAINS if d in res]))
    return res


def print_scores(res, title=''):
    for dom in list(DOMAINS) + ['ALL']:
        if dom in res:
            s = res[dom]
            print(f'--- {title} [{dom}] macro-F1 {s["macro_f1"]:.4f} | macro-P {s["macro_p"]:.4f} | macro-R {s["macro_r"]:.4f} | '
                  f'accuracy {s["accuracy"]:.4f} | {s["n_sentences"]} sentences')
    print(f'--- {title} selection score (mean domain macro-F1): {res["select"]:.4f}')


def print_per_label(res, title=''):
    print(f'per-role F1 ({title}):')
    print(f'   {"role":24s}' + ''.join(f'{d + " F1":>9s}{d + " sup":>8s}' for d in DOMAINS if d in res))
    for l in LABELS:
        print(f'   {l:24s}' + ''.join(f'{res[d]["per_label"][l]["f1"]:9.3f}{res[d]["per_label"][l]["support"]:8d}' for d in DOMAINS if d in res))


def confusion_report(res, top=8):
    """Most frequent (gold -> predicted) mistakes per domain: shows which closely related roles get mixed up."""
    out = {}
    for dom in list(DOMAINS) + ['ALL']:
        if dom not in res:
            continue
        cm = np.asarray(res[dom]['cm']); pairs = []
        for i in range(NUM_CLASSES):
            for j in range(NUM_CLASSES):
                if i != j and cm[i, j] > 0:
                    pairs.append((int(cm[i, j]), LABELS[i], LABELS[j], float(cm[i, j] / max(1, cm[i].sum()))))
        pairs.sort(reverse=True)
        out[dom] = [dict(count=c, gold=g, pred=p, share_of_gold=s) for c, g, p, s in pairs[:top]]
        print(f'[{dom}] most frequent confusions (count, gold -> predicted, share of that gold role):')
        for c, g, p, s in pairs[:top]:
            print(f'      {c:6d}  {g:24s} -> {p:24s} {100 * s:5.1f}%')
    return out

## 7. Load your Phase-1 model
The architecture code is copied unchanged from your Phase-1 notebook (collapsed); the weights come from your checkpoint.

In [20]:
# helper: Phase-1 model code, unchanged
import math
from dataclasses import dataclass
from types import SimpleNamespace
from typing import List, Optional
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

@dataclass
class Config:
    vocab_size: int = 32000
    d_model: int = 768
    n_layer: int = 12
    n_head: int = 12
    n_kv_head: Optional[int] = None
    max_seq_len: int = 2048
    rope_theta: float = 10000.0
    rope_pct: float = 1.0
    dropout: float = 0.0
    tie_embeddings: bool = False
    qk_norm: bool = False
    grad_ckpt: bool = False
    layer_types: Optional[List[str]] = None   # per layer: "r" linear attention, "n" global softmax attention (NoPE)
    la_decay: str = "fixed"              # "fixed" (RetNet) | "data" (GLA-style, input dependent)
    la_conv: int = 0                     # short causal depthwise conv kernel (0 = off)
    la_l2norm: bool = False
    la_rope: bool = True
    la_chunk: int = 64

    @property
    def head_dim(self):
        return self.d_model // self.n_head

    @property
    def kv_heads(self):
        return self.n_kv_head or self.n_head

    @property
    def ff_dim(self):
        return int(round(8 / 3 * self.d_model / 64)) * 64

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.w = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        xf = x.float()
        out = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (out * self.w.float()).type_as(x)


class Rotary(nn.Module):
    def __init__(self, dim, max_len, theta):
        super().__init__()
        inv = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / max(dim, 1)))
        freqs = torch.outer(torch.arange(max_len).float(), inv)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, T, start=0):
        """cos/sin for positions start .. start+T-1 (start > 0 only while generating with a cache)."""
        return self.cos[start:start + T], self.sin[start:start + T]


def apply_rope(x, cos, sin, rot_dim):
    """x: (B,H,T,hd). Rotates the first rot_dim dims (half-split convention)."""
    if rot_dim == 0:
        return x
    xr, xp = x[..., :rot_dim], x[..., rot_dim:]
    x1, x2 = xr.float().chunk(2, dim=-1)
    c, s = cos[None, None], sin[None, None]
    xr = torch.cat([x1 * c - x2 * s, x1 * s + x2 * c], dim=-1).to(x.dtype)
    return torch.cat([xr, xp], dim=-1)


class MLP(nn.Module):
    """SwiGLU feed-forward."""
    def __init__(self, cfg):
        super().__init__()
        d, f = cfg.d_model, cfg.ff_dim
        self.gate = nn.Linear(d, f, bias=False)
        self.up = nn.Linear(d, f, bias=False)
        self.down = nn.Linear(f, d, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

# --------------------------------------------------------------------------------------
# KV cache + attention
# --------------------------------------------------------------------------------------
class KVCache:
    """Pre-allocated key/value buffer for ONE attention layer (stored un-expanded, i.e. Hkv heads).
    Lets generation process one new token per step instead of re-running the whole prefix."""

    def __init__(self, max_len):
        self.max_len, self.n, self.k, self.v = max_len, 0, None, None

    def append(self, k, v):
        if self.k is None:
            B, H, _, hd = k.shape
            self.k = k.new_empty(B, H, self.max_len, hd)
            self.v = v.new_empty(B, H, self.max_len, hd)
        T = k.size(2)
        self.k[:, :, self.n:self.n + T] = k
        self.v[:, :, self.n:self.n + T] = v
        self.n += T

    def get(self, window=None):
        s = 0 if window is None else max(0, self.n - window)
        return self.k[:, :, s:self.n], self.v[:, :, s:self.n]


def sdpa_gqa(q, k, v, mask=None, causal=False, p=0.0):
    """q: (B,H,T,hd)   k,v: (B,Hkv,S,hd).  Handles GQA without copying the cache when T == 1."""
    B, H, T, hd = q.shape
    Hkv = k.size(1)
    if T == 1 and mask is None:                       # single-token decode step
        if Hkv != H:                                  # fold the query-head group into the "length" axis
            r = H // Hkv
            out = F.scaled_dot_product_attention(q.reshape(B, Hkv, r, hd), k, v)
            return out.reshape(B, H, 1, hd)
        return F.scaled_dot_product_attention(q, k, v)
    if Hkv != H:
        r = H // Hkv
        k, v = k.repeat_interleave(r, dim=1), v.repeat_interleave(r, dim=1)
    return F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=p,
                                          is_causal=causal and mask is None)

class Attention(nn.Module):
    """MHA/GQA self-attention with optional RoPE (or NoPE), QK-norm, value residual, sliding-window mask
    and KV cache.  Returns (output, v_of_the_new_tokens)."""

    def __init__(self, cfg, causal=True, use_rope=True, value_residual=False):
        super().__init__()
        d = cfg.d_model
        self.H, self.Hkv, self.hd = cfg.n_head, cfg.kv_heads, cfg.head_dim
        assert self.H % self.Hkv == 0, "n_head must be divisible by n_kv_head"
        self.q = nn.Linear(d, self.H * self.hd, bias=False)
        self.k = nn.Linear(d, self.Hkv * self.hd, bias=False)
        self.v = nn.Linear(d, self.Hkv * self.hd, bias=False)
        self.o = nn.Linear(self.H * self.hd, d, bias=False)
        self.q_norm = RMSNorm(self.hd) if cfg.qk_norm else None
        self.k_norm = RMSNorm(self.hd) if cfg.qk_norm else None
        self.p_drop = cfg.dropout
        self.causal = causal
        self.rot_dim = (int(self.hd * cfg.rope_pct) // 2 * 2) if use_rope else 0
        self.lam = nn.Parameter(torch.zeros(self.Hkv, 1, 1)) if value_residual else None

    def forward(self, x, rope=None, v_first=None, mask=None, cache=None, window=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.H, self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.Hkv, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.Hkv, self.hd).transpose(1, 2)
        if self.lam is not None and v_first is not None:
            lam = torch.sigmoid(self.lam).to(v.dtype)
            v = (1 - lam) * v + lam * v_first
        v_new = v
        if self.q_norm is not None:
            q, k = self.q_norm(q), self.k_norm(k)
        if self.rot_dim:
            cos, sin = rope
            q, k = apply_rope(q, cos, sin, self.rot_dim), apply_rope(k, cos, sin, self.rot_dim)
        causal = self.causal
        if cache is not None:
            cache.append(k, v)
            k, v = cache.get(window if T == 1 else None)   # window only matters for 1-token steps
            causal = T > 1                                  # prefill: causal; decode step: sees everything cached
        out = sdpa_gqa(q, k, v, mask=mask, causal=causal, p=self.p_drop if self.training else 0.0)
        return self.o(out.transpose(1, 2).reshape(B, T, self.H * self.hd)), v_new

class LMHead(nn.Module):
    """Final norm + unembedding (optionally tied) + optional multi-token-prediction heads."""

    def __init__(self, cfg, tok_emb: nn.Embedding):
        super().__init__()
        self.cfg = cfg
        self.norm = RMSNorm(cfg.d_model)
        self.proj = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.proj.weight = tok_emb.weight
        self.mtp = nn.ModuleList([nn.Linear(cfg.d_model, cfg.d_model, bias=False)
                                  for _ in range(getattr(cfg, "mtp_heads", 0))])

    def logits(self, h):
        return self.proj(self.norm(h))

    def forward(self, h, targets=None):
        h = self.norm(h)
        logits = self.proj(h)
        if targets is None:
            return logits, None
        V = logits.size(-1)
        loss = F.cross_entropy(logits.reshape(-1, V).float(), targets.reshape(-1), ignore_index=-100)
        self.main_loss = loss.detach()          # next-token loss only (without MTP terms), for logging
        w = getattr(self.cfg, "mtp_weight", 0.3)
        for k, p in enumerate(self.mtp, start=1):
            if targets.size(1) <= k:
                break
            hk = h[:, :-k] + F.silu(p(h[:, :-k]))
            lk = self.proj(hk)
            loss = loss + w * F.cross_entropy(lk.reshape(-1, V).float(), targets[:, k:].reshape(-1),
                                              ignore_index=-100)
        return logits, loss

def ckpt_call(fn, enabled, *args):
    return checkpoint(fn, *args, use_reentrant=False) if enabled else fn(*args)


def init_weights(model, depth):
    for m in model.modules():
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)
    std = 0.02 / math.sqrt(2 * depth)
    for n, p in model.named_parameters():
        if n.endswith(("o.weight", "down.weight")):
            nn.init.normal_(p, 0.0, std)


class BaseLM(nn.Module):
    """Common interface used by training.py / generation.py:
         forward(x, targets=None, src=None) -> (logits, loss)
         prefill(prompt_ids, max_new)       -> (logits_for_next_token, state)      [generation]
         step(token_ids, state)             -> logits_for_next_token               [generation]
       `eval_offset` = number of leading label positions that are NOT scored (encoder-decoder only)."""
    eval_offset = 0

    def num_params(self, non_embedding=False):
        n = sum(p.numel() for p in self.parameters())
        return n - self.tok.weight.numel() if non_embedding else n

def chunk_linear_attn(q, k, v, log_a, chunk=64, return_state=False):
    """q,k: (B,H,T,dk)  v: (B,H,T,dv)  log_a: (B,H,T) with log_a <= 0.
    Equivalent to  S_t = exp(log_a_t) S_{t-1} + k_t^T v_t ;  o_t = q_t S_t.
    With return_state=True also returns the final S (B,H,dk,dv) so generation can continue recurrently."""
    B, H, T, dk = q.shape
    dv = v.shape[-1]
    out_dtype = v.dtype
    q, k, v, log_a = q.float(), k.float(), v.float(), log_a.float()
    pad = (-T) % chunk
    if pad:
        q, k, v = (F.pad(t, (0, 0, 0, pad)) for t in (q, k, v))
        log_a = F.pad(log_a, (0, pad))                 # padded steps: decay 1, k = v = 0  -> state unchanged
    N = (T + pad) // chunk
    q = q.reshape(B, H, N, chunk, dk)
    k = k.reshape(B, H, N, chunk, dk)
    v = v.reshape(B, H, N, chunk, dv)
    b = log_a.reshape(B, H, N, chunk).cumsum(-1)                    # inclusive cumulative log-decay

    causal = torch.tril(torch.ones(chunk, chunk, dtype=torch.bool, device=q.device))
    decay = torch.exp((b.unsqueeze(-1) - b.unsqueeze(-2)).masked_fill(~causal, float("-inf")))
    intra = ((q @ k.transpose(-1, -2)) * decay) @ v                 # (B,H,N,C,dv)

    q_in = q * torch.exp(b).unsqueeze(-1)                           # decay from chunk start to i
    k_out = k * torch.exp(b[..., -1:] - b).unsqueeze(-1)            # decay from j to chunk end
    chunk_decay = torch.exp(b[..., -1])                             # (B,H,N)
    S = q.new_zeros(B, H, dk, dv)
    inter = []
    for n in range(N):
        inter.append(q_in[:, :, n] @ S)
        S = S * chunk_decay[:, :, n, None, None] + k_out[:, :, n].transpose(-1, -2) @ v[:, :, n]
    out = (intra + torch.stack(inter, 2)).reshape(B, H, N * chunk, dv)[:, :, :T].to(out_dtype)
    return (out, S) if return_state else out

class LinearState:
    """Recurrent state of one linear-attention layer while generating."""
    def __init__(self):
        self.S = None        # (B,H,dk,dv) float32
        self.conv = None     # (B,3*H*hd,K-1) last inputs of the causal conv


class LinearAttentionCore(nn.Module):
    """baseline  (la_decay="fixed"): RetNet-style multi-scale retention (fixed per-head decay, RoPE,
                 output gate, per-head norm).
       modified  (la_decay="data"):  input-dependent decay gate, short causal conv on q/k/v, L2-normalised
                 q/k, no RoPE (decay supplies position information)."""

    def __init__(self, cfg: Config):
        super().__init__()
        d, H, hd = cfg.d_model, cfg.n_head, cfg.head_dim
        self.H, self.hd, self.chunk = H, hd, cfg.la_chunk
        self.q, self.k, self.v, self.g = (nn.Linear(d, H * hd, bias=False) for _ in range(4))
        self.o = nn.Linear(H * hd, d, bias=False)
        self.onorm = RMSNorm(hd)
        self.l2norm = cfg.la_l2norm
        self.rot_dim = (int(hd * cfg.rope_pct) // 2 * 2) if cfg.la_rope else 0
        self.data_decay = cfg.la_decay == "data"
        if self.data_decay:
            self.a = nn.Linear(d, H)
            with torch.no_grad():                      # multi-scale init: fast ... slow forgetting
                self.a.bias.copy_(torch.linspace(0, 8, H))
        else:
            gamma = 1 - 2.0 ** (-5 - torch.arange(H).float())          # RetNet multi-scale decays
            self.register_buffer("log_gamma", gamma.log().view(1, H, 1), persistent=False)
        self.conv = None
        if cfg.la_conv > 0:
            ch = 3 * H * hd
            self.conv = nn.Conv1d(ch, ch, cfg.la_conv, groups=ch, padding=cfg.la_conv - 1)

    def _prep(self, q, k, rope, B, T):
        """reshape to heads, normalise, rotate"""
        H, hd = self.H, self.hd
        if self.l2norm:
            q, k = F.normalize(q, dim=-1), F.normalize(k, dim=-1)
        else:
            q = q * hd ** -0.5
        if self.rot_dim:
            cos, sin = rope
            q, k = apply_rope(q, cos, sin, self.rot_dim), apply_rope(k, cos, sin, self.rot_dim)
        return q, k

    def _log_a(self, x, B, T):
        if self.data_decay:
            return F.logsigmoid(self.a(x).float()).transpose(1, 2) / 16.0   # (B,H,T), in (-inf,0)
        return self.log_gamma.expand(B, self.H, T)

    def _finish(self, o, x, B, T):
        H, hd = self.H, self.hd
        o = self.onorm(o.transpose(1, 2).to(x.dtype))                      # per-head norm
        o = o * F.silu(self.g(x).view(B, T, H, hd))                        # output gate
        return self.o(o.reshape(B, T, H * hd))

class LinearAttention(LinearAttentionCore):
    """forward pass (chunkwise for training / prompt) + one-token recurrent step (generation)."""

    def forward(self, x, rope=None, cache: LinearState = None):
        B, T, _ = x.shape
        H, hd = self.H, self.hd
        q, k, v = self.q(x), self.k(x), self.v(x)
        if cache is not None and T == 1:
            return self._step(x, q, k, v, rope, cache)
        if self.conv is not None:
            cat = torch.cat([q, k, v], -1).transpose(1, 2)                 # (B,ch,T)
            if cache is not None:                                          # prefill: remember conv tail
                K = self.conv.kernel_size[0]
                tail = cat[..., -(K - 1):] if K > 1 else cat[..., :0]
                cache.conv = F.pad(tail, (K - 1 - tail.size(-1), 0))
            qkv = F.silu(self.conv(cat)[..., :T]).transpose(1, 2)
            q, k, v = qkv.split(H * hd, dim=-1)
        q, k, v = (t.reshape(B, T, H, hd).transpose(1, 2) for t in (q, k, v))
        q, k = self._prep(q, k, rope, B, T)
        log_a = self._log_a(x, B, T)
        if cache is not None:
            o, cache.S = chunk_linear_attn(q, k, v, log_a, self.chunk, return_state=True)
        else:
            o = chunk_linear_attn(q, k, v, log_a, self.chunk)              # (B,H,T,hd)
        return self._finish(o, x, B, T)

    def _step(self, x, q, k, v, rope, cache):
        """one-token recurrent update: S <- a*S + k^T v ; o = q S"""
        B, H, hd = x.size(0), self.H, self.hd
        if self.conv is not None:
            cur = torch.cat([q, k, v], -1).transpose(1, 2)                 # (B,ch,1)
            win = torch.cat([cache.conv.to(cur.dtype), cur], -1)           # (B,ch,K)
            cache.conv = win[..., 1:]
            y = (win * self.conv.weight.squeeze(1)).sum(-1, keepdim=True) + self.conv.bias[:, None]
            qkv = F.silu(y).to(x.dtype).transpose(1, 2)
            q, k, v = qkv.split(H * hd, dim=-1)
        q, k, v = (t.reshape(B, 1, H, hd).transpose(1, 2) for t in (q, k, v))
        q, k = self._prep(q, k, rope, B, 1)
        a = self._log_a(x, B, 1).exp().unsqueeze(-1)                       # (B,H,1,1)
        cache.S = cache.S * a + k.float().transpose(-1, -2) @ v.float()
        o = (q.float() @ cache.S).to(v.dtype)                              # (B,H,1,hd)
        return self._finish(o, x, B, 1)

class Block(nn.Module):
    """Pre-norm residual block with an injected token mixer (linear attention or softmax attention)."""

    def __init__(self, cfg, mixer: nn.Module):
        super().__init__()
        self.n1, self.n2 = RMSNorm(cfg.d_model), RMSNorm(cfg.d_model)
        self.mixer, self.mlp = mixer, MLP(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, rope=None, cache=None):
        h = self.mixer(self.n1(x), rope, cache=cache)
        if isinstance(h, tuple):                      # softmax attention returns (out, v)
            h = h[0]
        x = x + self.drop(h)
        return x + self.drop(self.mlp(self.n2(x)))

class DecoderLM(BaseLM):
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        assert len(cfg.layer_types) == cfg.n_layer, "layer_types must have n_layer entries"
        self.tok = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.rope = Rotary(int(cfg.head_dim * cfg.rope_pct) // 2 * 2, cfg.max_seq_len, cfg.rope_theta)
        self.blocks = nn.ModuleList([Block(cfg, LinearAttention(cfg) if t == "r" else
                                           Attention(cfg, use_rope=False)) for t in cfg.layer_types])
        self.head = LMHead(cfg, self.tok)
        init_weights(self, cfg.n_layer)

    def hidden(self, idx, caches=None, start=0):
        rope = self.rope(idx.shape[1], start)
        x = self.tok(idx)
        for i, blk in enumerate(self.blocks):
            if caches is None:
                x = ckpt_call(blk, self.cfg.grad_ckpt and self.training, x, rope)
            else:
                x = blk(x, rope, cache=caches[i])
        return x

    def forward(self, idx, targets=None, src=None):
        return self.head(self.hidden(idx), targets)

    # ---- generation: recurrent state for linear layers, KV cache for the softmax layers ----
    def prefill(self, idx, max_new):
        caches = [LinearState() if t == "r" else KVCache(idx.shape[1] + max_new) for t in self.cfg.layer_types]
        h = self.hidden(idx, caches=caches)
        return self.head.logits(h[:, -1]), SimpleNamespace(layers=caches, pos=idx.shape[1])

    def step(self, tok, state):
        h = self.hidden(tok, caches=state.layers, start=state.pos)
        state.pos += 1
        return self.head.logits(h[:, -1])

In [21]:
def find_ckpt():
    if PHASE1_CKPT and os.path.exists(PHASE1_CKPT):
        return PHASE1_CKPT
    cands = sorted(glob.glob('/kaggle/input/**/*.pt', recursive=True))
    norm = lambda s: s.lower().replace('-', '').replace('_', '')
    hit = [c for c in cands if norm(CKPT_HINT) in norm(c)] or cands
    if not hit:
        raise FileNotFoundError('No Phase-1 checkpoint found: attach it as a Kaggle dataset or set PHASE1_CKPT')
    print('checkpoint candidates:', *hit, sep='\n   ')
    return ([c for c in hit if 'best' in os.path.basename(c)] or [c for c in hit if 'last' in os.path.basename(c)] or hit)[0]

ckpt_path = find_ckpt()
PHASE1 = torch.load(ckpt_path, map_location='cpu', weights_only=False)     # {'model', 'cfg', 'step', 'val_loss'}
D_MODEL = PHASE1['cfg']['d_model']
ARCH_NAME = 'arch3_linear'
print('using', ckpt_path, '| Phase-1 step', PHASE1.get('step'), '| val loss', PHASE1.get('val_loss'))

checkpoint candidates:
   /kaggle/input/datasets/sunnysfhbv/model-arch-3-updated/best.pt
using /kaggle/input/datasets/sunnysfhbv/model-arch-3-updated/best.pt | Phase-1 step 1400 | val loss 2.829142426535975


In [22]:
# helper: rebuild the Phase-1 model from the checkpoint and expose its hidden states
def load_phase1_backbone(keep_layers=None):
    valid = {f.name for f in fields(Config)}
    cfg = Config(**{k: v for k, v in PHASE1['cfg'].items() if k in valid})
    cfg.grad_ckpt, cfg.dropout = GRAD_CKPT, 0.1
    m = DecoderLM(cfg)
    missing, unexpected = m.load_state_dict(PHASE1['model'], strict=False)
    assert not missing, f'missing keys when loading Phase-1 weights: {missing[:5]}'
    m.head.proj, m.head.mtp = nn.Identity(), nn.ModuleList()                # LM output layer / MTP heads are not needed
    return m


def backbone_features(m, ids, mask):
    """(batch, tokens) ids -> (batch, tokens, d_model). Causal model, right padding: padding never affects real tokens."""
    return m.head.norm(m.hidden(ids))


def get_blocks(m):
    return m.blocks


def get_final_norm(m):
    return m.head.norm

In [23]:
_probe = load_phase1_backbone()
print(f'{ARCH_NAME}: {sum(p.numel() for p in _probe.parameters()) / 1e6:.1f}M parameters loaded from Phase 1 (d_model {D_MODEL})')
del _probe

arch3_linear: 154.3M parameters loaded from Phase 1 (d_model 768)


## 8. Adaptation methods (choose with `METHOD` in the configuration cell)
The Phase-1 weights are frozen first; each method then makes a different small part trainable. The sentence head is always trained.

| `METHOD` | what is trained | idea |
|---|---|---|
| `'lorma'` | low-rank matrices, **multiplicative** (LoRMA+) | `W' = (I + (α/r)·B·A)·W0`, computed as `z = W0·x`, `y = z + (α/r)·B·(A·z)` |
| `'lora'` | low-rank matrices, additive | `W' = W0 + (α/r)·B·A` |
| `'qlora'` | LoRA matrices on a **4-bit NF4-quantised** frozen backbone | weights stored in 4 bit (block size 64, de-quantised on the fly), additive LoRA on top |
| `'full'` | every parameter | full fine-tuning (reference) |

`B` (LoRA / LoRMA / QLoRA) starts at zero, so every method starts exactly at the Phase-1 model.

In [24]:
# helper: adaptation methods
class LoRALinear(nn.Module):
    """mode 'lora':  y = W0 x + s*B*A*x          mode 'lorma' (LoRMA+):  y = z + s*B*A*z  with z = W0 x   (= (I + s*B*A)*W0*x)"""
    def __init__(self, base, r, alpha, mode, dropout=0.0):
        super().__init__()
        self.base, self.mode, self.scale = base, mode, alpha / r
        d_out, d_in = base.out_features, base.in_features
        self.A = nn.Parameter(torch.empty(r, d_in if mode == 'lora' else d_out))
        self.B = nn.Parameter(torch.zeros(d_out, r))              # B = 0 -> starts exactly at the Phase-1 model
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        if self.mode == 'lora':
            return self.base(x) + self.scale * F.linear(F.linear(self.drop(x), self.A), self.B)
        z = F.linear(x, self.base.weight)                         # W0 x
        z = z + self.scale * F.linear(F.linear(z, self.A), self.B)
        return z if self.base.bias is None else z + self.base.bias


NF4_TABLE = torch.tensor([-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635,
                          -0.18477343022823334, -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725,
                          0.24611230194568634, 0.33791524171829224, 0.44070982933044434, 0.5626170039176941,
                          0.7229568362236023, 1.0])


class NF4Linear(nn.Module):
    """Frozen linear layer stored in 4 bit (QLoRA): block-wise NF4 quantisation, two codes per byte, de-quantised on the fly."""
    def __init__(self, lin, block=64):
        super().__init__()
        w = lin.weight.data.float().to(device)
        self.out_features, self.in_features = w.shape
        assert w.numel() % block == 0, 'weight size must be a multiple of the block size'
        blocks = w.reshape(-1).view(-1, block)
        absmax = blocks.abs().amax(1, keepdim=True).clamp(min=1e-8)
        norm, table = blocks / absmax, NF4_TABLE.to(w.device)
        codes = torch.empty(blocks.shape, dtype=torch.uint8, device=w.device)
        for i in range(0, blocks.size(0), 4096):                  # nearest of the 16 NF4 levels
            codes[i:i + 4096] = (norm[i:i + 4096, :, None] - table).abs().argmin(-1).to(torch.uint8)
        self.register_buffer('packed', (codes[:, 0::2] << 4) | codes[:, 1::2])
        self.register_buffer('absmax', absmax.half())
        self.register_buffer('table', table, persistent=False)
        self.bias = None if lin.bias is None else nn.Parameter(lin.bias.data.clone().to(device), requires_grad=False)

    def dequant(self, dtype):
        codes = torch.stack([(self.packed >> 4).long(), (self.packed & 15).long()], -1).reshape(self.packed.size(0), -1)
        w = self.table[codes] * self.absmax.float()
        return w.reshape(self.out_features, self.in_features).to(dtype)

    def forward(self, x):
        return F.linear(x, self.dequant(x.dtype), self.bias)


def wrap_linears(module, targets, make):
    """Replace every nn.Linear called `targets` (q, k, v, o, ...) inside `module` by make(linear)."""
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in targets:
            setattr(module, name, make(child))
        else:
            wrap_linears(child, targets, make)


In [25]:
def apply_method(backbone, method):
    blocks = get_blocks(backbone)
    for p in backbone.parameters():
        p.requires_grad_(False)                                   # freeze Phase-1 first
    if method in ('lora', 'lorma'):
        wrap_linears(blocks, set(PEFT_TARGETS), lambda l: LoRALinear(l, LORA_R, LORA_ALPHA, method, LORA_DROPOUT))
    elif method == 'qlora':                                       # NF4-quantised frozen weights + additive LoRA
        wrap_linears(blocks, set(PEFT_TARGETS),
                     lambda l: LoRALinear(NF4Linear(l, QLORA_BLOCK), LORA_R, LORA_ALPHA, 'lora', LORA_DROPOUT))
    elif method == 'prefix':
        raise ValueError('prefix tuning needs softmax-attention layers everywhere; this model mostly uses linear attention')
    elif method == 'full':
        for p in backbone.parameters():
            p.requires_grad_(True)
    else:
        raise ValueError(f'unknown METHOD {method!r}')

## 9. Sentence classifier and training code
Phase-1 backbone + sentence head (sentence pooling, domain embedding, BiGRU over the sentences of a window, linear output).

In [26]:
class RRModel(nn.Module):
    """Phase-1 backbone -> one vector per sentence (mean of its token states + state at its last token)
       -> + domain embedding -> [BiGRU over the sentences of the window = document context] -> role logits."""
    def __init__(self, backbone, d_model, n_cls, n_dom, context, hidden=256, gru_layers=2):
        super().__init__()
        self.backbone, self.context, self.drop = backbone, context, nn.Dropout(0.1)
        h2 = 2 * hidden
        self.proj = nn.Sequential(nn.Linear(2 * d_model, h2), nn.GELU())
        self.dom_emb = nn.Embedding(n_dom, h2)
        if context == 'document':
            self.gru = nn.GRU(h2, hidden, num_layers=gru_layers, dropout=0.1 if gru_layers > 1 else 0.0,
                              batch_first=True, bidirectional=True)
            out_in = 2 * h2
        else:
            out_in = h2
        self.out = nn.Linear(out_in, n_cls)

    def forward(self, ids, mask, seg, last, smask, dom):
        with torch.autocast(device_type=device, dtype=amp_dtype, enabled=(device == 'cuda')):
            h = backbone_features(self.backbone, ids, mask)
        h = h.float()                                                              # (B, T, d)
        B, T, d = h.shape
        S = smask.size(1)
        oh = F.one_hot(seg.clamp(min=0), S).float() * (seg >= 0).unsqueeze(-1)    # (B, T, S) token -> sentence
        mean = torch.einsum('bts,btd->bsd', oh, h) / oh.sum(1).clamp(min=1).unsqueeze(-1)
        lastv = h.gather(1, last.unsqueeze(-1).expand(-1, -1, d))
        z = self.proj(self.drop(torch.cat([mean, lastv], -1)))
        z = self.drop(z + self.dom_emb(dom).unsqueeze(1))
        if self.context == 'document':
            lens = smask.sum(1).clamp(min=1).cpu()
            packed = nn.utils.rnn.pack_padded_sequence(z, lens, batch_first=True, enforce_sorted=False)
            g, _ = nn.utils.rnn.pad_packed_sequence(self.gru(packed)[0], batch_first=True, total_length=S)
            z = torch.cat([z, self.drop(g)], -1)
        return self.out(z)                                                         # (B, S, NUM_CLASSES)


def count_params(model):
    return sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad)


def build_rr(context, method, keep_layers=None):
    """Fresh Phase-1 backbone -> adaptation method -> sentence classification head."""
    backbone = load_phase1_backbone(keep_layers)
    apply_method(backbone, method)
    model = RRModel(backbone, D_MODEL, NUM_CLASSES, len(DOMAINS), context, HEAD_HIDDEN, GRU_LAYERS).to(device)
    total, trainable = count_params(model)
    print(f'{method} | context={context} | {total / 1e6:.1f}M parameters, {trainable / 1e6:.2f}M trainable ({100 * trainable / total:.2f}%)')
    return model, dict(method=method, context=context, total_params=total, trainable_params=trainable)

### 9a. Training loop pieces: optimizer, one epoch, prediction, early stopping
Loss = class-weighted cross-entropy with label smoothing. AdamW, warm-up then cosine decay, gradient clipping, fp16/bf16 autocast, best epoch kept by dev score.

In [27]:
# helper: optimizer, trainable-weight snapshots, one training epoch
def make_optimizer(model, lr, head_lr):
    is_head = lambda n: n.startswith(('out.', 'gru.', 'proj.', 'dom_emb.'))
    head = [p for n, p in model.named_parameters() if p.requires_grad and is_head(n)]
    body = [p for n, p in model.named_parameters() if p.requires_grad and not is_head(n)]
    groups = [dict(params=head, lr=head_lr, base_lr=head_lr)] + ([dict(params=body, lr=lr, base_lr=lr)] if body else [])
    return torch.optim.AdamW(groups, weight_decay=0.01)


def lr_factor(step, total, warmup):
    if step < warmup:
        return (step + 1) / warmup
    p = min(1.0, (step - warmup) / max(1, total - warmup))
    return 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * p))


def get_trainable_state(model):
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}


def load_trainable_state(model, state):
    params = dict(model.named_parameters())
    with torch.no_grad():
        for n, v in state.items():
            params[n].copy_(v)


def epoch_windows(docs, context, epoch):
    """Window boundaries are re-drawn every epoch (random core size per document) so the model sees different contexts."""
    rng, wins = np.random.RandomState(SEED + epoch), []
    for di, d in enumerate(docs):
        core = int(rng.randint(CORE_TOK // 2, CORE_TOK + 1)) if context == 'document' else 0
        wins += make_windows(di, d, context, core, CTX_TOK)
    rng.shuffle(wins)
    return wins


def train_epoch(wrap, opt, scaler, docs, wins, step, total_steps, bs, teacher=None, kd_alpha=0.5, kd_T=2.0):
    wrap.train()
    params = [p for g in opt.param_groups for p in g['params']]
    warmup, losses = max(1, int(0.06 * total_steps)), []
    for bi in tqdm(range(0, len(wins), bs), leave=False):
        batch = make_batch(docs, wins[bi:bi + bs])
        lab = batch[6].reshape(-1)
        logits = wrap(*batch[:6]).reshape(-1, NUM_CLASSES)
        loss = F.cross_entropy(logits, lab, weight=CLASS_W, ignore_index=-100, label_smoothing=LABEL_SMOOTH)
        if teacher is not None:                                              # knowledge distillation on the sentence logits
            with torch.no_grad():
                t_logits = teacher(*batch[:6]).reshape(-1, NUM_CLASSES)
            keep = lab != -100
            kd = F.kl_div(F.log_softmax(logits[keep] / kd_T, -1), F.softmax(t_logits[keep] / kd_T, -1),
                          reduction='batchmean') * kd_T ** 2
            loss = kd_alpha * loss + (1 - kd_alpha) * kd
        scaler.scale(loss / GRAD_ACCUM).backward()
        losses.append(float(loss.detach()))
        if (bi // bs + 1) % GRAD_ACCUM == 0 or bi + bs >= len(wins):
            for g in opt.param_groups:
                g['lr'] = g['base_lr'] * lr_factor(step, total_steps, warmup)
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
            step += 1
            if MAX_STEPS and step >= MAX_STEPS:
                return float(np.mean(losses)), step, True
    return float(np.mean(losses)), step, False


@torch.no_grad()
def predict_docs(wrap, docs, context, bs=None):
    """-> {uid: (n_sentences, NUM_CLASSES) probabilities}; every sentence is predicted exactly once (as a core sentence)."""
    wrap.eval()
    bs = bs or bsz(context) * 2
    probs = {d['uid']: np.zeros((len(d['tok']), NUM_CLASSES), np.float32) for d in docs}
    wins = []
    for di, d in enumerate(docs):
        wins += make_windows(di, d, context, CORE_TOK, CTX_TOK)
    wins.sort(key=lambda w: sum(len(t) for t in docs[w[0]]['tok'][w[1]:w[3]]))
    for i in range(0, len(wins), bs):
        chunk = wins[i:i + bs]
        pr = F.softmax(wrap(*make_batch(docs, chunk)[:6]).float(), -1).cpu().numpy()
        for j, (di, s, c, e) in enumerate(chunk):
            probs[docs[di]['uid']][c:e] = pr[j, c - s:e - s]
    return probs


def run_training(model, tag, train_docs, dev_docs, context, *, epochs, lr, head_lr=1e-3, teacher=None, kd_alpha=0.5, kd_T=2.0):
    """Trains `model`, keeps the best epoch on dev (mean domain macro-F1, early stopping), restores it, returns a summary."""
    wrap = nn.DataParallel(model) if N_GPU > 1 else model
    opt = make_optimizer(model, lr, head_lr)
    scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda' and amp_dtype == torch.float16))
    bs = bsz(context)
    steps_per_epoch = max(1, math.ceil(len(epoch_windows(train_docs, context, 0)) / (bs * GRAD_ACCUM)))
    total_steps = min(steps_per_epoch * epochs, MAX_STEPS or 10 ** 9)
    best, history, bad, step, t0 = dict(score=-1.0, epoch=0, state=None), [], 0, 0, time.time()
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    print(f'[{tag}] {steps_per_epoch} optimizer steps per epoch, {total_steps} in total')
    for ep in range(1, epochs + 1):
        wins = epoch_windows(train_docs, context, ep)
        loss, step, stop = train_epoch(wrap, opt, scaler, train_docs, wins, step, total_steps, bs, teacher, kd_alpha, kd_T)
        row = dict(epoch=ep, train_loss=round(loss, 4))
        res = evaluate_docs(dev_docs, predict_docs(wrap, dev_docs, context))
        row.update({f'dev_f1_{d}': round(res[d]['macro_f1'], 4) for d in DOMAINS if d in res}, dev_select=round(res['select'], 4))
        improved = res['select'] > best['score']
        if improved:
            best.update(score=res['select'], epoch=ep, state=get_trainable_state(model))
        bad = 0 if improved else bad + 1
        history.append(row)
        print(f'[{tag}]', row, '  <- best' if improved else '')
        if stop or bad >= PATIENCE:
            break
    load_trainable_state(model, best['state'])
    peak = torch.cuda.max_memory_allocated() / 1e9 if device == 'cuda' else 0.0
    print(f'[{tag}] finished in {(time.time() - t0) / 60:.1f} min | best epoch {best["epoch"]} | best dev score {best["score"]:.4f} | peak GPU memory {peak:.1f} GB')
    return dict(history=history, best_epoch=best['epoch'], best_score=best['score'], seconds=time.time() - t0, peak_mem_gb=peak)

## 10. Evaluation and submission helpers

In [28]:
# helper: validation report + test predictions -> submission files
def evaluate_report(model, tag, context, docs=None):
    """Scores on the development documents: per domain, per role, error analysis."""
    docs = docs if docs is not None else dev_docs
    wrap = nn.DataParallel(model) if N_GPU > 1 else model
    t0 = time.time()
    probs = predict_docs(wrap, docs, context)
    n_tok = sum(len(t) for d in docs for t in d['tok'])
    tok_per_s = n_tok / max(time.time() - t0, 1e-9)
    res = evaluate_docs(docs, probs)
    print_scores(res, tag)
    print_per_label(res, tag)
    conf = confusion_report(res)
    print(f'[{tag}] inference speed {tok_per_s:.0f} tokens/s')
    return dict(scores=res, confusions=conf, infer_tok_per_s=tok_per_s)


def predict_test(model, context):
    wrap = nn.DataParallel(model) if N_GPU > 1 else model
    probs = predict_docs(wrap, test_docs, context)
    return {d['uid']: [LABELS[i] for i in probs[d['uid']].argmax(1)] for d in test_docs}


def write_submission(pred, out_dir, prefix=''):
    """prediction.json = {document_id: [label per sentence]} (+ one file per domain). Lengths are verified."""
    per_dom, valid = defaultdict(dict), set(LABELS)
    for d in test_docs:
        labs = pred[d['uid']]
        assert len(labs) == len(d['sents']), (d['uid'], len(labs), len(d['sents']))
        assert all(l in valid for l in labs)
        per_dom[d['domain']][d['id']] = labs
    ids = [k for dom in per_dom.values() for k in dom]
    collide = len(ids) != len(set(ids))
    merged = {(f'{dom}_{k}' if collide else k): v for dom, docs_ in per_dom.items() for k, v in docs_.items()}
    files = {f'{prefix}prediction.json': merged}
    files.update({f'{prefix}{dom}_prediction.json': v for dom, v in per_dom.items()})
    for name, obj in files.items():
        json.dump(obj, open(os.path.join(out_dir, name), 'w', encoding='utf-8'), ensure_ascii=False)
    with zipfile.ZipFile(os.path.join(out_dir, f'{prefix}submission.zip'), 'w', zipfile.ZIP_DEFLATED) as z:
        for name in files:
            z.write(os.path.join(out_dir, name), arcname=name)
    print(f'wrote {list(files)} + {prefix}submission.zip:', {k: len(v) for k, v in per_dom.items()}, 'documents',
          '(document ids repeat across domains -> merged file uses "<DOMAIN>_<id>" keys)' if collide else '')
    d0 = test_docs[0]
    print('example:', d0['uid'], list(zip(pred[d0['uid']][:3], [s[:60] for s in d0['sents'][:3]])))

In [29]:
def summary_table(rows):
    """rows = [(name, info, train_result, validation_result)] -> compact comparison used for the report"""
    print(f'{"experiment":22s} {"params":>8s} {"trainable":>10s} ' + ''.join(f'{"F1 " + d:>8s}' for d in DOMAINS)
          + f'{"mean F1":>9s} {"macro-P":>8s} {"macro-R":>8s} {"tok/s":>8s} {"min":>6s} {"GPU GB":>7s}')
    for name, inf, tr, v in rows:
        s = v['scores']
        print(f'{name:22s} {inf["total_params"] / 1e6:7.1f}M {inf["trainable_params"] / 1e6:9.2f}M '
              + ''.join(f'{s[d]["macro_f1"]:8.4f}' for d in DOMAINS) + f'{s["select"]:9.4f} {s["ALL"]["macro_p"]:8.4f} '
              f'{s["ALL"]["macro_r"]:8.4f} {v["infer_tok_per_s"]:8.0f} {tr["seconds"] / 60:6.1f} {tr["peak_mem_gb"]:7.1f}')

## 11. Run the experiment
### 11a. Build the model

In [30]:
model, info = build_rr(CONTEXT, METHOD)

lorma | context=document | 165.9M parameters, 11.62M trainable (7.00%)


### 11b. Train

In [31]:
train_res = run_training(model, RUN_NAME, train_docs, dev_docs, CONTEXT, epochs=EPOCHS, lr=LR, head_lr=HEAD_LR)

[p3_arch3_lorma] 79 optimizer steps per epoch, 711 in total


  0%|          | 0/294 [00:00<?, ?it/s]

[p3_arch3_lorma] {'epoch': 1, 'train_loss': 2.3766, 'dev_f1_CL': 0.3965, 'dev_f1_IT': 0.4389, 'dev_select': 0.4177}   <- best


  0%|          | 0/308 [00:00<?, ?it/s]

[p3_arch3_lorma] {'epoch': 2, 'train_loss': 1.7621, 'dev_f1_CL': 0.4403, 'dev_f1_IT': 0.4331, 'dev_select': 0.4367}   <- best


  0%|          | 0/302 [00:00<?, ?it/s]

... (9 outputs left out)

  0%|          | 0/302 [00:00<?, ?it/s]

[p3_arch3_lorma] {'epoch': 8, 'train_loss': 0.8591, 'dev_f1_CL': 0.481, 'dev_f1_IT': 0.5141, 'dev_select': 0.4976} 


  0%|          | 0/314 [00:00<?, ?it/s]

[p3_arch3_lorma] {'epoch': 9, 'train_loss': 0.8381, 'dev_f1_CL': 0.4797, 'dev_f1_IT': 0.5255, 'dev_select': 0.5026} 
[p3_arch3_lorma] finished in 46.0 min | best epoch 6 | best dev score 0.5083 | peak GPU memory 4.2 GB


### 11c. Validate: scores per domain, per role, error analysis

In [32]:
val_res = evaluate_report(model, RUN_NAME, CONTEXT)

--- p3_arch3_lorma [CL] macro-F1 0.4716 | macro-P 0.4791 | macro-R 0.5392 | accuracy 0.5301 | 1396 sentences
--- p3_arch3_lorma [IT] macro-F1 0.5450 | macro-P 0.5838 | macro-R 0.5309 | accuracy 0.5952 | 746 sentences
--- p3_arch3_lorma [ALL] macro-F1 0.5098 | macro-P 0.5257 | macro-R 0.5237 | accuracy 0.5528 | 2142 sentences
--- p3_arch3_lorma selection score (mean domain macro-F1): 0.5083
per-role F1 (p3_arch3_lorma):
   role                        CL F1  CL sup    IT F1  IT sup
   Fact                        0.638     249    0.768     195
   Issue                       0.286       3    0.696      12
   ArgumentPetitioner          0.463     344    0.554      56
   ArgumentRespondent          0.517     236    0.536      36
   Statute                     0.772      58    0.772      62
   Dissent                     0.000       0    0.000       0
   PrecedentReliedUpon         0.409     142    0.643     112
   PrecedentNotReliedUpon      0.000       0    0.424      50
   PrecedentOverrul

### 11d. Predict the test documents and write the submission

In [33]:
write_submission(predict_test(model, CONTEXT), OUTPUT_DIR)

wrote ['prediction.json', 'CL_prediction.json', 'IT_prediction.json'] + submission.zip: {'CL': 5, 'IT': 5} documents 
example: CL::HC_Raghupati_Singhania__vs__Competition_Commission_ofDE201616051616063761COM513489 [('Fact', 'Manmohan, J. 1 .'), ('Fact', 'Present writ petition has been filed, challenging the notice'), ('Fact', 'The CCI has further directed individuals who have been ident')]


### 11e. Optional: sentence-only vs document-context ablation
Set `RUN_CONTEXT_ABLATION = True` in the configuration to train the *other* context mode with the same method and compare. (The submission above always comes from `CONTEXT`.)

In [34]:
ablation = None
if RUN_CONTEXT_ABLATION:
    other = 'sentence' if CONTEXT == 'document' else 'document'
    model_b, info_b = build_rr(other, METHOD)
    res_b = run_training(model_b, f'{RUN_NAME}_{other}', train_docs, dev_docs, other, epochs=EPOCHS, lr=LR, head_lr=HEAD_LR)
    val_b = evaluate_report(model_b, f'{RUN_NAME}_{other}', other)
    ablation = {other: dict(info=info_b, train=res_b, validation=val_b)}
    summary_table([(f'{METHOD} / {CONTEXT}', info, train_res, val_res), (f'{METHOD} / {other}', info_b, res_b, val_b)])
    del model_b; gc.collect(); torch.cuda.empty_cache()
else:
    summary_table([(f'{METHOD} / {CONTEXT}', info, train_res, val_res)])

experiment               params  trainable    F1 CL   F1 IT  mean F1  macro-P  macro-R    tok/s    min  GPU GB
lorma / document         165.9M     11.62M   0.4716  0.5450   0.5083   0.5257   0.5237     8645   46.0     4.2


## 12. Save weights and results

In [35]:
torch.save(dict(trainable=get_trainable_state(model), method=METHOD, context=CONTEXT, labels=LABELS, domains=DOMAINS,
                core_tok=CORE_TOK, ctx_tok=CTX_TOK, phase1_ckpt=ckpt_path), os.path.join(OUTPUT_DIR, f'{RUN_NAME}_trainable.pt'))
json.dump(dict(run=RUN_NAME, arch=ARCH_NAME, adaptation=info, train=train_res, validation=val_res, context_ablation=ablation),
          open(os.path.join(OUTPUT_DIR, 'results.json'), 'w'), indent=1, default=str)
print('saved:', sorted(os.listdir(OUTPUT_DIR)))

saved: ['CL_prediction.json', 'IT_prediction.json', 'p3_arch3_lorma_trainable.pt', 'prediction.json', 'results.json', 'submission.zip']
